# CML3112 Machine Learning: Engineering ML project, REHEARSAL (unmarked)
**This is a rehearsal. Nothing here is marked or submitted.** It has the same structure, the same task wording and the same kinds of check as the marked project, on the same steel plant file, but with a different target: here you predict **the lagging reactive power the plant draws in a 15-minute reading** (kVarh), with its own warning limit. The marked project predicts the energy used (kWh). Your numbers here will not be the numbers you get in the project. Released Tuesday 17 November 2026, right after Class 3. The project window opens Monday 23 November 2026, 00:00 and closes Wednesday 25 November 2026, 23:59 Kampala time.

**How to use it.** Open it in Colab (File, then Save a copy in Drive). Work through it as if it were the real project: set a timer for 3 hours if you want to practise the timing. Use the self-check cells. Ask about anything unclear on the class channel before the window. A solved walk-through is not given: the point is that you can do it yourself.

This project uses only what you learned in Weeks 1 to 10 (up to Class 3). You work on real readings from a steel plant, `Steel_industry_data.csv` (DAEWOO Steel, Gwangyang, South Korea, 2018, one reading every 15 minutes), and answer two questions: **Question 1** how much is the lagging reactive power the plant draws in a 15-minute reading (kVarh) in months the model has never seen, and **Question 2** which readings should raise a warning. November and December are the test set and are opened **once**.

Data: Sathishkumar V E, Changsun Shin and Yongyun Cho (2021). Steel Industry Energy Consumption [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52G8C. Licence CC BY 4.0.

## How to work
1. Fill in the identity cell below (in the issued notebook it is filled in for you; check it) and run it. Then run the supplied cell **Prepare the data and the libraries**. The data file is stored inside it, so you need no upload and no download.
2. Work from top to bottom. Replace each `TODO` and each `None`. Keep the function names, the argument order and the variable names exactly as printed. Run each cell and read what it prints. You may add your own cells. Your work is marked wherever it is, as long as the required function names exist when the notebook runs from the top.
3. Run the **self-check** cells to see which functions still need work. They are not marked.
4. Part B is a form (the `MEMO` cell). Nothing is written as an essay.
5. Choose **Runtime, then Restart session and run all**. Fix any error, then save with **File, then Save**.
6. Run the last cell, click **Submit test**, open the form link, check your details and press **Submit** there. Submission is finished only when the form says your response has been recorded.

## Time plan (180 minutes)
| Step | Time |
|---|---|
| Setup: open your notebook, run the first cells | 10 min |
| Tasks 1 and 2: clean the rows, split by month | 35 min |
| Task 3: the three models | 30 min |
| Task 4: warnings and the cost table | 25 min |
| Task 5: cross-validation by month | 15 min |
| Tasks 6 and 7: choose a model, open the test set once | 10 min |
| Task 8: the Part B memo form | 25 min |
| Check, restart and run all, submit | 20 min |
| Spare time (breaks, reconnecting) | 10 min |

## How it is marked
Everything is marked by the lecturer's computer. No person reads your notebook for marks. Your functions are run on **other rows that you cannot see** (hidden checks), and a few of your results on the file are compared with the correct values. Of the 30 Part A marks, 21 are for the functions, 3 for your results, and 6 for how you worked: the test set opened once and only after you chose, no data leakage (fit preprocessing on the training set only), the same answer every time (keep `random_state=17`), and readable charts. Part B gives 15 marks for the memo form. A list of notebooks that look unusual is checked by the lecturer afterwards.

## Help during the window
Ask wording and technical questions on the class channel, 08:00 to 20:00 each day of the window (lecturer to confirm these hours). If something fails (Colab, the entry page, the form), report it at once on the class channel with a screenshot. The lecturer can authorise one fresh attempt, still ending at the window's close.

**Rules (the same rules as Practical Test 1).** Individual work, online, from anywhere. **Permitted resources:** your lecture slides, your lab notebooks from Classes 1 to 3, the revision guide, the cheatsheet and your own course notes. Write the code yourself.
**No AI or outside help:** do not use Gemini, ChatGPT, Copilot or any other AI tool to generate, complete, rewrite, explain or debug your answers. Do not get answers from another person, collaborate or copy someone else's work.
**Understanding check:** after the window, any student may be asked, in an online call, to explain their submitted code or make a small change to it, without AI or another person's help. The copy checks and the identity checks run on every notebook.
**Declaration:** by submitting, I confirm that I completed this project myself without AI assistance or another person's help, using only the permitted resources. I can explain and change the code I submitted.


In [ ]:
FULL_NAME = ""
EMAIL = ""
STUDENT_ID = ""   # your Student Registration Number exactly as on your university record
COHORT = "CML3112"  # Do not edit: identifies your course for marking.
TEST_ID = "ML_PROJECT_REHEARSAL"  # Do not edit: identifies this assessment.


## Your data
Real readings from the electricity meters of a steel plant (DAEWOO Steel, Gwangyang, South Korea), one row every 15 minutes for the whole of 2018: 35,040 rows. Source: Sathishkumar V E, Changsun Shin and Yongyun Cho (2021). Steel Industry Energy Consumption [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52G8C. Licence CC BY 4.0. The same file is used by every student.

| Column | Meaning |
|---|---|
| date | Day and time of the reading, written DD/MM/YYYY HH:MM. Each day's last reading is written with that day's date and 00:00 (see Task 1). |
| Usage_kWh | Energy used in the 15 minutes, kWh. |
| Lagging_Current_Reactive.Power_kVarh, Leading_Current_Reactive_Power_kVarh | Reactive power in the 15 minutes, kVarh. |
| CO2(tCO2) | Carbon dioxide, calculated from the energy used (it leaks the answer; Task 1 removes it). |
| Lagging_Current_Power_Factor, Leading_Current_Power_Factor | Power factors in percent, 0 to 100. Calculated from the energy and the reactive power, so they are not used as features. |
| NSM | Seconds since midnight (0 to 85500). |
| WeekStatus, Day_of_week | Weekday or Weekend, and the day's name. |
| Load_Type | Light_Load, Medium_Load or Maximum_Load: a time-of-day band. Not used. |

The target is `Lagging_Current_Reactive.Power_kVarh` (kVarh). The features are `FEATURES` = `Usage_kWh`, `Leading_Current_Reactive_Power_kVarh`, `NSM`, `weekend` (`weekend` is made in Task 1). A **record** is one row as a dictionary. A **list of records** is several rows. Your functions take and return these. Missing values are `None`. Calculate from the lists you are given. Do not use the global tables inside a function, because the marked checks give your functions other rows. The constants `TARGET`, `FEATURES`, `LIMIT` and the others from the setup cell may be used.

In [ ]:
#@title Prepare the data and the libraries - Do not edit this cell
# IUEA-SETUP-CELL  (supplied: do not edit)
# Data: Steel Industry Energy Consumption, DAEWOO Steel, Gwangyang, South Korea, 2018, one reading every 15 minutes.
# Sathishkumar V E, Changsun Shin and Yongyun Cho (2021), UCI Machine Learning Repository, https://doi.org/10.24432/C52G8C. Licence CC BY 4.0.
# The file is stored inside this cell (compressed), so nothing is downloaded. It is written out and checked with SHA-256.
import base64, bz2, hashlib, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

DATA_FILE = 'Steel_industry_data.csv'
DATA_SHA256 = '9b1cee6f9cb9cd9df2b95814ca90a9a2ff15b7f5f1fba0fae3c643e82072eacc'
_PACKED = """
...
"""
def _data_is_intact():
    return os.path.exists(DATA_FILE) and hashlib.sha256(open(DATA_FILE, "rb").read()).hexdigest() == DATA_SHA256
if not _data_is_intact():
    _raw = bz2.decompress(base64.b64decode("".join(_PACKED.split())))
    if hashlib.sha256(_raw).hexdigest() != DATA_SHA256:
        raise RuntimeError("The data stored in this cell is damaged. Do not edit this cell. Tell your lecturer.")
    with open(DATA_FILE, "wb") as _f:
        _f.write(_raw)

TARGET = 'Lagging_Current_Reactive.Power_kVarh'          # Question 1: the lagging reactive power the plant draws in a 15-minute reading
TARGET_UNIT = 'kVarh'
FEATURES = ['Usage_kWh', 'Leading_Current_Reactive_Power_kVarh', 'NSM', 'weekend']
# the power factors are left out: they are calculated from the energy and the reactive power, so they would leak the answer; CO2(tCO2) is removed in Task 1 for the same reason
LIMIT = 40.0          # a reading is a WARNING (1) when its target is at or above this limit, in kVarh (invented teaching number)
MAX_MAE = 6.0         # the client accepts a typical prediction error (MAE) of at most this many kVarh (invented teaching number)
MAX_MS = 50               # and a prediction time of at most this many milliseconds (invented teaching number)
LATENCY_MS = {'baseline': 1, 'linear': 3, 'tree': 30}   # invented teaching numbers
MISSED_COST, FALSE_ALARM_COST = 5, 1   # invented teaching costs (a false negative costs 5, a false positive costs 1)

def records_from_table(table):
    """A list of dictionaries, one per row. Missing values (NaN) become None."""
    objects = table.astype(object)
    return objects.where(table.notna(), None).to_dict("records")

raw_table = pd.read_csv(DATA_FILE, encoding="utf-8-sig")    # the file starts with a byte-order mark: utf-8-sig removes it
all_records = records_from_table(raw_table)
development_records = [r for r in all_records if int(r["date"][3:5]) <= 10]   # January to October. November and December are the TEST SET.
print("Rows in the file:", len(raw_table))
print("Rows you may develop with (January to October):", len(development_records))
print(raw_table.head())


## Task 1: Clean the rows and look at them (3 automatic marks, Weeks 3 to 7)

A **record** is one row written as a dictionary, with the column names of the file as keys. A missing value is `None`.

What is really in this file (checked before the project): no missing cells and no repeated rows, but three things you must handle.
- Each day's **last** reading is written with that day's date and the time `00:00` (for example `01/01/2018 00:00` comes straight after `01/01/2018 23:45`). Its real time is midnight at the **end** of that day, so one day must be added to its timestamp.
- One reading (7 November) has every measurement 0: the meter recorded nothing.
- `CO2(tCO2)` is calculated from `Usage_kWh`, so keeping it would leak the answer (data leakage).

`clean_rows(records)` returns a cleaned **copy** of the records, in the same order. For each record:
- make `timestamp`, a text `"YYYY-MM-DD HH:MM"` from `date` (format `"%d/%m/%Y %H:%M"`), with one day added when the time is 00:00; make `month`, the month **as written** in `date` (1 to 12); make `weekend`, `1` when `WeekStatus` is `"Weekend"`, else `0`;
- if the timestamp was already seen, skip the row (keep only the first);
- drop the row if `TARGET` is `None`, or if `Usage_kWh` and both power factors are all 0 (nothing recorded), or if a value is impossible: `Usage_kWh` or either reactive power below 0, a power factor below 0 or above 100 (0 and 100 are valid), `NSM` below 0 or above 85500;
- remove the key `CO2(tCO2)`. Do not fill a missing value: it stays `None`, and the models do the imputation later from the training set only.

The marking checks give your function other rows that contain these problems and others, so write every rule even if this file does not need it. Never change the list you were given. Then use your function on the development rows and draw **Chart 1**: `Usage_kWh` across, `Lagging_Current_Reactive.Power_kVarh` up, with a title, both axis names and the unit (kVarh).

In [ ]:
def clean_rows(records):
    """Return a cleaned copy of the records (see the task text above)."""
    # TODO 1: make an empty list called kept and an empty set called seen.
    # TODO 2: loop over records. when = datetime.strptime(record["date"], "%d/%m/%Y %H:%M"); month = when.month
    #         if when.hour == 0 and when.minute == 0: when = when + timedelta(days=1)
    #         timestamp = when.strftime("%Y-%m-%d %H:%M"). If timestamp is in seen, skip the row (continue); otherwise add it to seen.
    # TODO 3: skip the row if record[TARGET] is None, if nothing was recorded, or if a value is impossible (see the task text).
    # TODO 4: row = a copy of record without "CO2(tCO2)"; add row["timestamp"], row["month"] and row["weekend"]; add row to kept.
    return None   # TODO 5: return kept instead


In [ ]:
development_rows = None      # TODO: call clean_rows on development_records
print("Rows kept:", None)    # TODO: replace None with the number of rows in development_rows

figure, axes = plt.subplots()
# TODO: axes.scatter(...): FEATURES[0] on the x axis, TARGET on the y axis, small points (s=2) (hint: table = pd.DataFrame(development_rows))
# TODO: axes.set_title(...), axes.set_xlabel(...) and axes.set_ylabel(...). Write the unit in the y label.
chart_1 = figure
plt.show()


## Task 2: Split by month (2 automatic marks, Week 10)

`time_split(records)` receives cleaned records and returns a dictionary with three lists of `timestamp` values, each in the order the records came in:
- `"train"`: months 1 to 8, January to August (the **training set**);
- `"validation"`: months 9 and 10, September and October (the **validation set**);
- `"test"`: months 11 and 12, November and December (the **test set**).

**Why not a random split?** Readings 15 minutes apart are almost the same. A random split puts near-copies of the same hour in the training set and the validation set, and lets the model learn from days that come after the ones it is tested on. The score would look better than it will be on next month. Splitting by month keeps whole months apart and keeps the future in the future. Your development rows have no November or December rows, so their `"test"` list is empty. That is correct.

When you pick the training and validation rows, look the timestamps up in a **set**. Looking each one up in a list of 23,000 timestamps makes the cell run for minutes, and the marking computer stops a cell after 30 seconds.

In [ ]:
def time_split(records):
    """Return {"train": [...], "validation": [...], "test": [...]} (lists of timestamp, in input order)."""
    # TODO: for each record, look at record["month"] and put record["timestamp"] in the right list.
    return None


In [ ]:
split_ids = None     # TODO: time_split(development_rows)
train_ids, valid_ids = None, None   # TODO: set(split_ids["train"]) and set(split_ids["validation"]) (a set makes the look-up fast)
train_rows = None    # TODO: the development rows whose timestamp is in train_ids
valid_rows = None    # TODO: the development rows whose timestamp is in valid_ids
print("Training rows:", None, "| validation rows:", None)   # TODO: replace the Nones with the two counts


## Task 3: Question 1: a baseline model, a linear regression and a decision tree (6 automatic marks, Weeks 8 and 9)

**Question 1.** How much is the lagging reactive power the plant draws in a 15-minute reading, in kVarh, for months the model has never seen?

`fit_predict_regression(training_records, feature_records)` trains (fits) three models on the training records and predicts for the feature records. Use the columns in `FEATURES` (`Usage_kWh`, `Leading_Current_Reactive_Power_kVarh`, `NSM`, `weekend`), in that order. The feature records have **no target**.
- every model is a pipeline that first does **imputation** of missing values with the **median of the training rows** (`SimpleImputer(strategy="median")`), then fits;
- `"baseline"`: the baseline model, which always predicts the mean training target (`DummyRegressor`);
- `"linear"`: multiple linear regression, `LinearRegression`;
- `"tree"`: a decision tree, `DecisionTreeRegressor(max_depth=4, random_state=17)`. The `max_depth` is a **hyperparameter**: a setting you choose before training.

Return a dictionary with three lists of predictions, in the order of the feature records. The baseline model must use the **training** rows only: a baseline built with validation or test targets has **leaked**.

`regression_errors(actual, predicted)` returns `{"mae": ..., "rmse": ...}` over the same pairs, not rounded. A **residual** is actual minus predicted. MAE (mean absolute error) is the mean of the absolute residuals. RMSE (root mean squared error) is the square root of the mean of the squared residuals.

In [ ]:
def fit_predict_regression(training_records, feature_records):
    """Return {"baseline": [...], "linear": [...], "tree": [...]}."""
    # TODO 1: table = pd.DataFrame(training_records); new = pd.DataFrame(feature_records)
    # TODO 2: X_train = table[FEATURES]; y_train = table[TARGET]; X_new = new[FEATURES].
    # TODO 3: a dictionary of three pipelines. Each is make_pipeline(SimpleImputer(strategy="median"), <the model>).
    # TODO 4: loop over the dictionary: fit on (X_train, y_train), predict X_new, and store the predictions as a list of floats under the model's name.
    return None


In [ ]:
def regression_errors(actual, predicted):
    """Return {"mae": ..., "rmse": ...}, not rounded."""
    # TODO: use numpy: residuals = np.array(actual) - np.array(predicted); MAE uses np.abs and the mean; RMSE uses squares, the mean, then np.sqrt.
    return None


In [ ]:
# Evaluate the three models on the validation set (September and October). Keep the names: validation_predictions, validation_mae and validation_rmse.
validation_features = None    # TODO: the validation rows without the TARGET key
validation_actual = None      # TODO: the list of validation targets
validation_predictions = None # TODO: fit_predict_regression(train_rows, validation_features)
validation_mae = None         # TODO: a dictionary {"baseline": ..., "linear": ..., "tree": ...} of MAE values, from regression_errors
validation_rmse = None        # TODO: the same with RMSE
print("Validation MAE  (" + TARGET_UNIT + "):", validation_mae)
print("Validation RMSE (" + TARGET_UNIT + "):", validation_rmse)

figure, axes = plt.subplots()
# TODO: a bar chart of the three MAE values, with a title, a y label that includes the unit, and the model names on the x axis.
chart_2 = figure
plt.show()


## Task 4: Question 2: which readings should raise a warning? (6 automatic marks, Weeks 9 and 10)

**Question 2.** A reading is a **warning** (1, the positive class) when its `Lagging_Current_Reactive.Power_kVarh` is at or above `LIMIT` (40 kVarh) and **no warning** (0, the negative class) below it. A high lagging reactive power is when the plant is most likely to pay a power-factor penalty. The limit is an invented teaching number. Which readings should raise a warning?

`warning_probabilities(training_records, feature_records, limit)` returns the probability of a warning, for each feature record, from a pipeline of `SimpleImputer(strategy="median")`, `StandardScaler()` and `LogisticRegression(max_iter=1000)` (binary classification), fitted on the training records only. The label is `1` when the training target is `>= limit`. Use the columns in `FEATURES`.

`classification_scores(actual, probability, threshold)` predicts a warning when the probability is `>= threshold` (the **decision threshold**), then returns a dictionary with the confusion matrix counts `tp`, `fp`, `fn`, `tn` (true positives, false positives, false negatives, true negatives) and `precision`, `recall`, `f1`, `cost` and `auc`. A false negative (a missed warning) costs `MISSED_COST` and a false positive (a false alarm) costs `FALSE_ALARM_COST` (invented teaching costs), so `cost = 5 * fn + 1 * fp`. When a ratio has a zero denominator, use 0. `auc` is the ROC AUC of the probabilities and does not depend on the threshold.

Then try decision thresholds 0.3, 0.5 and 0.7 on the validation set, and keep the one with the lowest cost as `chosen_threshold` (if two tie, take the higher one).

In [ ]:
def warning_probabilities(training_records, feature_records, limit):
    """Return a list of warning probabilities, one per feature record."""
    # TODO 1: tables as before; y_train = (training target >= limit) turned into 0 and 1 (astype(int)).
    # TODO 2: model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000)); fit it on table[FEATURES].
    # TODO 3: predict_proba(new[FEATURES]) gives two columns: column 1 is the probability of a warning. Return it as a list of floats.
    return None


In [ ]:
def classification_scores(actual, probability, threshold):
    """Return tp, fp, fn, tn, precision, recall, f1, cost and auc."""
    # TODO 1: predicted = 1 where the probability is at or above the threshold, else 0 (a list comprehension).
    # TODO 2: tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()   (see Class 2)
    # TODO 3: precision_score, recall_score and f1_score with zero_division=0; roc_auc_score(actual, probability).
    # TODO 4: cost = MISSED_COST * fn + FALSE_ALARM_COST * fp. Turn numpy numbers into int() and float() before returning.
    return None


In [ ]:
warning_actual = None        # TODO: 1 or 0 for each validation row: is the target at or above LIMIT?
warning_probability = None   # TODO: warning_probabilities(train_rows, validation_features, LIMIT)
threshold_table = {}         # TODO: for each threshold in 0.3, 0.5, 0.7: threshold_table[threshold] = classification_scores(...)
warning_auc = None           # TODO: the "auc" value of any one entry of threshold_table
chosen_threshold = None      # TODO: the threshold with the lowest cost (if two tie, the higher threshold)
for threshold, scores in threshold_table.items():
    print(threshold, scores)
print("Chosen threshold:", chosen_threshold)


## Task 5: Can you trust it? Cross-validation by month (4 automatic marks, Week 10)

One validation set of two months can mislead. **Cross-validation by month** (group K-fold cross-validation where each group is one month, the idea of `GroupKFold`) gives each month a turn as the unseen month.

`month_cv_mae(records)` receives clean records with **no missing values**. Hold out each month from 1 to 10 in turn (January to October; months 11 and 12 are never used). For each fold, fit `LinearRegression` on `FEATURES` using the other nine months, and compute the MAE on the held-out month. Return `{"folds": [...], "mean": ..., "spread": ...}`: the ten MAEs in month order, their mean, and their sample standard deviation (`ddof=1`). Do not round.

**Data leakage check, in your own head:** the imputer, the StandardScaler (standardisation) and the baseline model's mean are all fitted on the training rows only. If any of them had seen validation or test rows, the scores would look better than they are.

In [ ]:
def month_cv_mae(records):
    """Return {"folds": [ten MAEs, January to October], "mean": ..., "spread": ...}."""
    # TODO 1: table = pd.DataFrame(records); keep only the rows whose month is 1 to 10.
    # TODO 2: for month in range(1, 11): train = the rows of the OTHER months, test = the rows of this month.
    # TODO 3: fit LinearRegression on train[FEATURES]; the MAE on test is np.mean(np.abs(...)).
    # TODO 4: mean = np.mean(folds); spread = np.std(folds, ddof=1).
    return None


In [ ]:
complete_rows = None    # TODO: the development rows with no missing value in FEATURES or TARGET
cv_summary = None       # TODO: month_cv_mae(complete_rows)
print("Cross-validated MAE (" + TARGET_UNIT + "):", cv_summary)


## Task 6: Choose one model with the supplied rule (supplied rule; no marks of its own)

The model-choice rule is supplied (the cell below says Do not edit this cell). Run it, then use it. The rule: keep the models whose validation MAE is at most `MAX_MAE` **and** whose prediction time is at most `MAX_MS` (both limits included); among those, choose the smallest validation MAE; break a tie by the shorter time, then by the name. If no model qualifies, the rule returns `None`. Never use the test set to choose.

The client for this project is the plant's energy manager. The client accepts a typical prediction error of at most `MAX_MAE` (6 kVarh) and a prediction time of at most `MAX_MS` (50 ms). Both numbers are invented for teaching.

Your part: build `candidate_table`, one dictionary per model with `name`, `validation_mae` (your own Task 3 value) and `latency_ms` (from `LATENCY_MS`), and call `choose_model(candidate_table, MAX_MAE, MAX_MS)`.

In [ ]:
#@title The model-choice rule - Do not edit this cell
# IUEA-CHOOSE-CELL  (supplied: do not edit)
def choose_model(candidates, max_mae, max_ms):
    """Keep the candidates with validation_mae <= max_mae and latency_ms <= max_ms. Choose the smallest validation_mae;
    break a tie by the smaller latency_ms, then by the name. Return the chosen name, or None if nobody qualifies."""
    allowed = [c for c in candidates if c["validation_mae"] <= max_mae and c["latency_ms"] <= max_ms]
    if not allowed:
        return None
    return sorted(allowed, key=lambda c: (c["validation_mae"], c["latency_ms"], c["name"]))[0]["name"]


In [ ]:
candidate_table = []     # TODO: one dictionary per model (baseline, linear, tree): name, validation_mae (from your Task 3 results), latency_ms (from LATENCY_MS)
chosen_model = None      # TODO: choose_model(candidate_table, MAX_MAE, MAX_MS)
print(candidate_table)
print("Chosen model:", chosen_model)


## Task 7: Open the test set, once (2 automatic marks, Week 10)

November and December are the **test set**: two months that nothing so far has used. They give **one** honest estimate for the model you chose.

How it works. The supplied cell below is marked Do not edit this cell. Run it once to define `open_test_set(model_name, threshold)`. Then call it **one time**, with the model you chose in Task 6 and the decision threshold you chose in Task 4. It cleans the whole file with your `clean_rows`, trains your functions on all the development months (January to October), evaluates on November and December, prints the result and writes your choice to a small file. If you call it again, the notebook counts a second opening: you lose the "opened once" mark even when the choice is the same, and you lose more if the choice is different.

You must not use November or December to choose a model, a threshold or anything else, and you must not look at them before this call. Report whatever the result is in your Part B form.

In [ ]:
#@title Open the test set (run it ONCE, at the very end) - Do not edit this cell
# IUEA-TEST-CELL  (supplied: do not edit)
import json as _json
TEST_OPEN_LOG = []
TEST_OUTCOMES = []
_TEST_FILE = "test_first_opening.json"

def open_test_set(model_name, threshold):
    """Evaluate ONE chosen model and ONE chosen decision threshold on the test set (November and December), with the functions you wrote."""
    if model_name not in ("baseline", "linear", "tree"):
        raise ValueError('model_name must be "baseline", "linear" or "tree"')
    if not isinstance(threshold, (int, float)) or not 0 < threshold < 1:
        raise ValueError("threshold must be a number between 0 and 1")
    first = _json.load(open(_TEST_FILE)) if os.path.exists(_TEST_FILE) else None
    if first is None:
        _json.dump({"model": model_name, "threshold": threshold}, open(_TEST_FILE, "w"))
    TEST_OPEN_LOG.append([model_name, threshold])
    changed = first is not None and (first["model"] != model_name or first["threshold"] != threshold)
    if changed:
        print("WARNING: THE TEST SET WAS ALREADY OPENED with model", first["model"], "and threshold", first["threshold"],
              "- opening it again with a different choice means it is no longer an honest test.")
    cleaned = clean_rows(all_records)
    development = [r for r in cleaned if r["month"] <= 10]
    held_out = [r for r in cleaned if r["month"] >= 11]
    features = [{k: v for k, v in r.items() if k != TARGET} for r in held_out]
    actual = [r[TARGET] for r in held_out]
    errors = regression_errors(actual, fit_predict_regression(development, features)[model_name])
    probability = warning_probabilities(development, features, LIMIT)
    warning = classification_scores([1 if v >= LIMIT else 0 for v in actual], probability, threshold)
    outcome = {"model": model_name, "threshold": threshold, "n_rows": len(held_out), "mae": errors["mae"], "rmse": errors["rmse"],
               "warning": warning, "changed_after_opening": bool(changed)}
    TEST_OUTCOMES.append(outcome)
    print("Test set opened with", model_name, "and threshold", threshold, "on", len(held_out), "rows")
    print("  MAE", round(errors["mae"], 3), TARGET_UNIT, "| RMSE", round(errors["rmse"], 3), TARGET_UNIT)
    print("  warnings: precision", round(warning["precision"], 3), "recall", round(warning["recall"], 3), "F1", round(warning["f1"], 3), "AUC", round(warning["auc"], 3))
    return outcome


In [ ]:
test_result = None     # TODO: open_test_set(chosen_model, chosen_threshold)


## Self-check (not marked)
Run this supplied cell to see which functions still need work. It uses tiny examples with known answers. The marked checks use other rows.

In [ ]:
# This cell is a SELF-CHECK. It is not marked. It uses tiny examples with known answers, so you can see at once if a function is wrong.
# The marked (hidden) checks run your functions on OTHER rows you cannot see, so passing here does not mean full marks.
def _try(label, test):
    try:
        ok = bool(test())
    except Exception as error:
        ok = False
    print(("PASS  " if ok else "TODO  ") + label)
    return ok

def _row(date, usage, lag=2.0, lead=0.0, lpf=90.0, dpf=100.0, week="Weekday"):
    h, m = int(date[11:13]), int(date[14:16])
    return {"date": date, "Usage_kWh": usage, "Lagging_Current_Reactive.Power_kVarh": lag, "Leading_Current_Reactive_Power_kVarh": lead,
            "CO2(tCO2)": 0.0, "Lagging_Current_Power_Factor": lpf, "Leading_Current_Power_Factor": dpf, "NSM": h * 3600 + m * 60,
            "WeekStatus": week, "Day_of_week": "Monday", "Load_Type": "Light_Load"}
_small = [_row("01/01/2018 23:45", 3.0), _row("01/01/2018 00:00", 3.5), _row("01/01/2018 00:00", 9.9), _row("02/01/2018 00:15", 4.0),
          _row("02/01/2018 00:30", 0.0, 0.0, 0.0, 0.0, 0.0), _row("09/09/2018 10:00", 50.0, week="Weekend"), _row("20/11/2018 12:00", 40.0, lpf=101.0)]
_small[3][TARGET] = None
_ts = ["2018-01-01 23:45", "2018-01-02 00:00", "2018-09-09 10:00"]
_checks = []
_checks.append(_try("clean_rows keeps 3 rows, in order, with the midnight reading moved to the next day",
                    lambda: [r["timestamp"] for r in clean_rows(_small)] == _ts))
_checks.append(_try("clean_rows adds month (as written) and weekend, and removes CO2(tCO2)",
                    lambda: [r["month"] for r in clean_rows(_small)] == [1, 1, 9] and [r["weekend"] for r in clean_rows(_small)] == [0, 0, 1]
                    and all("CO2(tCO2)" not in r for r in clean_rows(_small))))
_checks.append(_try("clean_rows does not change the list you give it", lambda: (len(_small) == 7 and "CO2(tCO2)" in _small[0] and clean_rows(_small) is not _small)))
_checks.append(_try("time_split puts January in train, September in validation",
                    lambda: time_split(clean_rows(_small)) == {"train": _ts[:2], "validation": _ts[2:], "test": []}))
_toy = [dict({f: float(i) for f in FEATURES}, **{TARGET: 2.0 * i, "month": 1, "timestamp": "2018-01-0%d 10:00" % i}) for i in range(1, 7)]
_new = [dict({f: 3.5 for f in FEATURES}, **{FEATURES[0]: None, "month": 9, "timestamp": "2018-09-01 10:00"})]
_checks.append(_try("fit_predict_regression returns baseline, linear and tree, one prediction per feature record",
                    lambda: sorted(fit_predict_regression(_toy, _new)) == ["baseline", "linear", "tree"] and all(len(v) == 1 for v in fit_predict_regression(_toy, _new).values())))
_checks.append(_try("the baseline model predicts the mean of the TRAINING targets (7.0)", lambda: abs(fit_predict_regression(_toy, _new)["baseline"][0] - 7.0) < 1e-9))
_checks.append(_try("regression_errors([1, 3], [2, 5]) has mae 1.5 and rmse sqrt(2.5)",
                    lambda: abs(regression_errors([1, 3], [2, 5])["mae"] - 1.5) < 1e-9 and abs(regression_errors([1, 3], [2, 5])["rmse"] - 2.5 ** 0.5) < 1e-9))
_checks.append(_try("classification_scores on a known case (tp 1, fp 1, fn 1, tn 1, cost 6, auc 0.75)",
                    lambda: (lambda s: (s["tp"], s["fp"], s["fn"], s["tn"], s["cost"], round(s["auc"], 2)) == (1, 1, 1, 1, 6, 0.75))(classification_scores([1, 0, 1, 0], [0.9, 0.8, 0.2, 0.1], 0.5))))
_checks.append(_try("warning_probabilities gives numbers between 0 and 1",
                    lambda: all(0 <= p <= 1 for p in warning_probabilities(_toy, _new, 8.0))))
_checks.append(_try("month_cv_mae returns ten folds, a mean and a spread",
                    lambda: len(month_cv_mae(complete_rows)["folds"]) == 10 and {"mean", "spread"} <= set(month_cv_mae(complete_rows))))
for _name in ["validation_mae", "cv_summary", "warning_auc", "candidate_table", "chosen_model", "chosen_threshold"]:
    _checks.append(_try("the variable " + _name + " has been calculated (it is not None)", lambda: globals().get(_name) is not None))
_checks.append(_try("the test set has been opened exactly once", lambda: len(TEST_OPEN_LOG) == 1))
_checks.append(_try("Chart 1 and Chart 2 each have a title, named axes and the unit",
                    lambda: all(f.axes[0].get_title() and f.axes[0].get_xlabel() and TARGET_UNIT.lower() in f.axes[0].get_ylabel().lower() for f in (chart_1, chart_2))))
print()
print(sum(_checks), "of", len(_checks), "self-checks pass. Now choose Runtime, Restart session and run all, and check that nothing stops with an error.")


## Task 8: Part B, the decision memo form (15 automatic marks)
Your memo is a **form**. You write it for **the plant's energy manager**. Fill in the dictionary `MEMO` in the next cell. Every value is checked by the marking computer; nothing is read by a person. Use your own numbers, with at least two decimals, in **kVarh**.

| Field | What to enter | Marks |
|---|---|---|
| `recommended_model`, `recommended_threshold` | One of the models you trained (`"baseline"`, `"linear"`, `"tree"`) and one decision threshold (0.3, 0.5 or 0.7). They must be the model and threshold your rule and cost table chose. | 3 |
| `validation_mae` | The validation MAE of each of the three models, as your notebook computed them. | 3 |
| `test_mae`, `test_recall` | The MAE and the warning recall on the test set from your ONE opening (recall is between 0 and 1). The marking computer recomputes them from your own functions. | 2 |
| `cv_mean_mae` | The mean MAE from your cross-validation by month. | 1 |
| `limits` | The letters of the **three** statements below that are **true limits of this study**. Three different letters. One mark for each true one. | 3 |
| `next_step` | One letter: the best next step. | 1 |
| `answer_choice`, `answer_test_set` | Two short written answers of 8 to 80 words each. One mark each. | 2 |

**Limits.** Choose the three statements that are TRUE and are limits of this study (some statements are false):
- **A.** RMSE is always smaller than MAE, so MAE is the more severe measure and should be the only one reported.
- **B.** The validation set is only two months (September and October), so the validation errors could change a lot with two other months.
- **C.** The test set is only November and December of the same year, so the test metrics are one estimate that may be luckier or unluckier than the truth.
- **D.** The baseline model should be left out of the comparison because it uses no features.
- **E.** The test MAE proves that the model will be this accurate in every month of every year.
- **F.** Because the test set was opened only once, the test metrics have no uncertainty at all.
- **G.** The model with the lowest training error is always the best model for new months.
- **H.** The client limits, the warning limit, the decision threshold options and the costs (5 and 1) are invented teaching numbers; a real client would set their own.
- **I.** All the readings come from one steel plant in one year (2018), so the model may not hold for another plant or for later years.
- **J.** Fitting the imputer on the training set only makes the validation results unreliable.

**Next step.** Choose the one best next step:
- **A.** Deploy the chosen model now, because the test MAE is below the client limit.
- **B.** Remove the baseline model from the report so the results look better.
- **C.** Tune the maximum depth of the decision tree until its test MAE is lowest, then report that test MAE.
- **D.** Collect readings from more years (or another plant) and check the chosen model again with cross-validation by month before any decision.
- **E.** Move the test months into the training set so the model learns from more rows, and report the training error.

**Written answers.** Write in your own words, in full sentences.
- `answer_choice`: In one or two sentences: why do you recommend this model? Use a number from the validation set and say what you compared it with.
- `answer_test_set`: In one or two sentences: why is the test set opened only once, at the very end, and never used to choose the model?

Each course has its own lettering of these lists.

In [ ]:
#@title Part B: the decision memo form (fill in every field; do not rename a field)
MEMO = {
    "recommended_model": "",                  # "baseline", "linear" or "tree"
    "recommended_threshold": None,            # 0.3, 0.5 or 0.7
    "validation_mae": {"baseline": None, "linear": None, "tree": None},   # your validation MAE values, in kVarh
    "cv_mean_mae": None,                      # your mean cross-validated MAE, in kVarh
    "test_mae": None,                         # the test MAE from your ONE opening, in kVarh
    "test_recall": None,                      # the test recall from your ONE opening (0 to 1)
    "limits": ["", "", ""],                   # three different letters from the Limits list
    "next_step": "",                          # one letter from the Next step list
    "answer_choice": "",                      # one or two sentences
    "answer_test_set": "",                    # one or two sentences
}


## Memo self-check (not marked)
Run this supplied cell after you fill in the form. It tells you which fields are empty and whether your numbers agree with your own notebook.

In [ ]:
# MEMO SELF-CHECK (not marked): did you fill in every field, and do your numbers agree with your own notebook?
# It does not tell you whether your choices are the best ones. The marking computer checks that.
def _memo_try(label, test):
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("PASS  " if ok else "TODO  ") + label)
    return ok

def _near(a, b):
    return isinstance(a, (int, float)) and isinstance(b, (int, float)) and abs(a - b) <= 0.011

_memo_checks = [
    _memo_try("recommended_model is one of the models you trained", lambda: MEMO["recommended_model"] in validation_mae),
    _memo_try("recommended_threshold is 0.3, 0.5 or 0.7", lambda: MEMO["recommended_threshold"] in (0.3, 0.5, 0.7)),
    _memo_try("validation_mae has your three validation MAE values (two decimals are enough)", lambda: all(_near(MEMO["validation_mae"][k], validation_mae[k]) for k in ("baseline", "linear", "tree"))),
    _memo_try("cv_mean_mae is your cross-validated mean MAE", lambda: _near(MEMO["cv_mean_mae"], cv_summary["mean"])),
    _memo_try("test_mae is the test MAE you saw when you opened the test set", lambda: _near(MEMO["test_mae"], test_result["mae"])),
    _memo_try("test_recall is the test recall you saw (between 0 and 1)", lambda: _near(MEMO["test_recall"], test_result["warning"]["recall"])),
    _memo_try("limits has three different letters", lambda: len({str(x).strip().upper() for x in MEMO["limits"] if str(x).strip()}) == 3),
    _memo_try("next_step is one letter", lambda: len(str(MEMO["next_step"]).strip()) == 1),
    _memo_try("answer_choice has 8 to 80 words", lambda: 8 <= len(str(MEMO["answer_choice"]).split()) <= 80),
    _memo_try("answer_test_set has 8 to 80 words", lambda: 8 <= len(str(MEMO["answer_test_set"]).split()) <= 80),
]
print()
print(sum(_memo_checks), "of", len(_memo_checks), "memo self-checks pass.")


## End of the rehearsal
Nothing is submitted from this notebook. In the project, the notebook has a last cell that saves a snapshot and opens the submission form, and a countdown at the top.

Before the window, check that you can: finish every function and see PASS in the self-check; run **Runtime, Restart session and run all** with no error; fill in every `MEMO` field and see 10 of 10 memo self-checks pass; and explain, in your own words, what each function does and why the test set is opened only once.